In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
LANDING = f"/Volumes/{CATALOG}/bronze/landing/sgs"
CONTROLE = f"/Volumes/{CATALOG}/bronze/landing/_controle/sgs"
TABELA = f"{CATALOG}.bronze.sgs_series"

(
    spark.readStream.format("cloudFiles")            # Auto Loader
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CONTROLE}/schema")
    .option("cloudFiles.inferColumnTypes", "false")  # tudo string na Bronze
    .load(LANDING)
    .withColumn("_arquivo_origem", F.col("_metadata.file_path"))
    .withColumn("_ingerido_em", F.current_timestamp())
    .writeStream
    .option("checkpointLocation", f"{CONTROLE}/checkpoint")
    .trigger(availableNow=True)                      # processa o que há de novo e para
    .toTable(TABELA)
    .awaitTermination()
)

display(
    spark.table(TABELA)
    .groupBy("serie_nome")
    .agg(F.count("*").alias("registros"))
)